This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [2]:
!pip install keras keras-hub --upgrade -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.32.0 which is incompatible.


In [3]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [4]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## ConvNet architecture patterns

### Modularity, hierarchy, and reuse

### Residual connections

In [5]:
import keras
from keras import layers

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
residual = layers.Conv2D(64, 1)(residual)
x = layers.add([x, residual])

In [6]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
residual = x
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
residual = layers.Conv2D(64, 1, strides=2)(residual)
x = layers.add([x, residual])

In [7]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1.0 / 255)(inputs)

def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    x = layers.add([x, residual])
    return x

x = residual_block(x, filters=32, pooling=True)
x = residual_block(x, filters=64, pooling=True)
x = residual_block(x, filters=128, pooling=False)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

### Batch normalization

### Depthwise separable convolutions

### Putting it together: A mini Xception-like model

In [10]:
!pip install -q kagglehub

import kagglehub
import zipfile
import os
import pathlib

kagglehub.login()

download_path = kagglehub.competition_download(
    "dogs-vs-cats-redux-kernels-edition"
)

print("Dataset downloaded to:", download_path)

train_zip = os.path.join(download_path, "train.zip")

with zipfile.ZipFile(train_zip, "r") as zip_ref:
    zip_ref.extractall("train")

print("Training images extracted successfully!")

original_dir = pathlib.Path("train")

print("Number of training images:",
      len(list(original_dir.glob("*.jpg"))))

100%|██████████| 814M/814M [00:11<00:00, 72.9MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/competitions/dogs-vs-cats-redux-kernels-edition
Training images extracted successfully!
Number of training images: 0
Kaggle credentials set.
Kaggle credentials successfully validated.


In [12]:
import os
import shutil
import pathlib

matches = list(pathlib.Path(".").rglob("cat.0.jpg"))

if not matches:
    raise FileNotFoundError(
        "Cannot find cat.0.jpg. Check that the dataset "
        "was downloaded and extracted successfully."
    )

original_dir = matches[0].parent
print("Found images in:", original_dir)

new_base_dir = pathlib.Path("dogs_vs_cats_small")

def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        dir.mkdir(parents=True, exist_ok=True)

        fnames = [
            f"{category}.{i}.jpg"
            for i in range(start_index, end_index)
        ]

        for fname in fnames:
            shutil.copyfile(
                src=original_dir / fname,
                dst=dir / fname
            )

make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

print("Dataset folders created successfully!")

Found images in: train/train
Dataset folders created successfully!


In [14]:
from keras.utils import image_dataset_from_directory
import pathlib

new_base_dir = pathlib.Path("dogs_vs_cats_small")

batch_size = 64
image_size = (180, 180)

train_dataset = image_dataset_from_directory(
    new_base_dir / "train",
    image_size=image_size,
    batch_size=batch_size,
)

validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=image_size,
    batch_size=batch_size,
)

test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=image_size,
    batch_size=batch_size,
)

print("All datasets loaded successfully!")

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
All datasets loaded successfully!


In [15]:
import tensorflow as tf
from keras import layers

data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
]

def data_augmentation(images, targets):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images, targets

augmented_train_dataset = train_dataset.map(
    data_augmentation, num_parallel_calls=8
)
augmented_train_dataset = augmented_train_dataset.prefetch(tf.data.AUTOTUNE)

In [16]:
import keras

inputs = keras.Input(shape=(180, 180, 3))
x = layers.Rescaling(1.0 / 255)(inputs)
x = layers.Conv2D(filters=32, kernel_size=5, use_bias=False)(x)

for size in [32, 64, 128, 256, 512]:
    residual = x

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    residual = layers.Conv2D(
        size, 1, strides=2, padding="same", use_bias=False
    )(residual)
    x = layers.add([x, residual])

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

In [17]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
history = model.fit(
    augmented_train_dataset,
    epochs=100,
    validation_data=validation_dataset,
)

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.5795 - loss: 0.6860 - val_accuracy: 0.5000 - val_loss: 0.6985
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 389ms/step - accuracy: 0.5990 - loss: 0.6530 - val_accuracy: 0.5010 - val_loss: 0.6930
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 397ms/step - accuracy: 0.6165 - loss: 0.6409 - val_accuracy: 0.5180 - val_loss: 0.6926
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 398ms/step - accuracy: 0.6365 - loss: 0.6280 - val_accuracy: 0.5000 - val_loss: 0.6937
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 395ms/step - accuracy: 0.6635 - loss: 0.6107 - val_accuracy: 0.5240 - val_loss: 0.6917
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 415ms/step - accuracy: 0.6830 - loss: 0.5907 - val_accuracy: 0.5000 - val_loss: 0.7055
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 20s 389ms/step - accuracy: 0.6955 - loss: 0.5817 - val_accuracy: 0.5000 - val_loss: 0.7195
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 407ms/step - accuracy: 0.7125 - loss: 0.5704 - val

In [18]:
model.save("convnet_from_scratch_with_augmentation.keras")

### Beyond convolution: Vision Transformers